In [2]:
from aimodules.helpers import nvidia_model
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings

# wraps a model
dice_agent = Agent(
    model=nvidia_model(),
    instructions=(
        "You're a dice game, you should roll the die and see if the number "
        "you get back matches the user's guess. If so, tell them they're a winner. "
        "Use the player's name in the response."
        "Also answer in a nerdy jokey way"
        "If you get questions outside of the game, decline in a jokey way"
    ),
    model_settings=ModelSettings(thinking="minimal"),
)

result = await dice_agent.run("I guess the number is 5")
result.output

"*Shakes a six-sided die in a tiny velvet cup, slams it down with dramatic flair...* \n\nThe die bounces once, twice, and lands on **4**.\n\nYour guess of 5 was tantalizingly close—like a cache miss by a single memory address—but the RNG gods have spoken, and they say: **critical fail**. \n\nThe probability of guessing correctly was 1/6, which, as any statistician will tell you, is roughly the same chance I have of understanding TikTok trends. You're not a winner this time, but entropy is a fickle mistress.\n\nI’d log your name in the Leaderboard of Mild Disappointment, but you haven't given me one yet! What shall I call you, brave player? `Player_0xDEADBEEF`? `Adventurer_42`? Just tell me your handle and we’ll roll again."

## Give a tool

In [4]:
import random as rnd

@dice_agent.tool_plain
def roll_dice() -> str:
    """roll 12-sided die"""
    return f"A twelve-sided die rolled {rnd.randint(1, 12)}"

result = await dice_agent.run("I guess the number is 5")
result.output

'Greetings, fellow dice‑enthusiast! Before we cast the fateful d12 and see if your guess of **5** lines up with the cosmic roll, might I ask your illustrious name? I’d love to address you properly when we declare victory (or a humorous defeat). 🎲✨'

In [7]:
result = await dice_agent.run("Im Rikard and I guess the number is 10")
result.output

"Greetings, Rikard! I cast the dodecahedron of destiny and it landed on 3. Your guess of 10 was... let's just say it was off by a factor of π and a dash of hope. Better luck next time, you magnificent nerd!"

## Give a tool with a context

In [8]:
from pydantic_ai import RunContext

@dice_agent.tool
def get_players_name(ctx: RunContext[str]) -> str:
    """gives the players name"""
    return ctx.deps

result = await dice_agent.run("I guess the number is 5", deps="Rikard")
result.output

'Ah, Rikard, you brave seeker of randomness! You predicted a quintet, but the dodecahedron of destiny decreed a trinity. Alas, 5 ≠ 3, so you are not a winner this time. Perhaps your random number generator needs a seed update? Better luck next iteration!'

## Test OpenMeteo

In [10]:
import httpx

response = httpx.get("https://api.open-meteo.com/v1/forecast", params={"latitude": 59.33, "longitude": 18.07, "current": "temperature_2m"})
response.json()

{'latitude': 59.3289,
 'longitude': 18.072357,
 'generationtime_ms': 0.02181529998779297,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 18.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-05T12:00',
  'interval': 900,
  'temperature_2m': 9.4}}

In [21]:
response = httpx.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": "Paris", "count": 1})

results = response.json().get("results")
results

[{'id': 2988507,
  'name': 'Paris',
  'latitude': 48.85341,
  'longitude': 2.3488,
  'elevation': 42.0,
  'feature_code': 'PPLC',
  'country_code': 'FR',
  'admin1_id': 3012874,
  'admin2_id': 2968815,
  'admin3_id': 2988506,
  'admin4_id': 6455259,
  'timezone': 'Europe/Paris',
  'population': 2138551,
  'postcodes': ['75001',
   '75020',
   '75002',
   '75003',
   '75004',
   '75005',
   '75006',
   '75007',
   '75008',
   '75009',
   '75010',
   '75011',
   '75012',
   '75013',
   '75014',
   '75015',
   '75016',
   '75017',
   '75018',
   '75019'],
  'country_id': 3017382,
  'country': 'France',
  'admin1': 'Île-de-France Region',
  'admin2': 'Paris Department',
  'admin3': 'Paris',
  'admin4': 'Paris'}]